In [1]:
from pyspark.sql import SparkSession
from datetime import datetime


# Initialize SparkSession
ss = SparkSession.builder.getOrCreate()

stations_df = ss.read.load("data/sampleData/stations.csv", format="csv", header=True, inferSchema=True, delimiter="\\t")
stations_df.createOrReplaceTempView("stations")

register_df = ss.read.load("data/sampleData/register.csv", format="csv", header=True, inferSchema=True, delimiter="\\t")
register_df.createOrReplaceTempView("register")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 13:26:38 WARN Utils: Your hostname, cubone, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlan0)
26/05/25 13:26:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 13:26:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ss.sql(
    "SELECT station, date_format(timestamp,'EE') as weekday, hour(timestamp) as hour, COUNT(free_slots) as critical_count \
    FROM register \
    WHERE NOT (used_slots = 0 AND free_slots = 0) \
    AND free_slots = 0 \
    GROUP BY station, weekday, hour"
).createOrReplaceTempView("critical")

In [3]:
ss.sql(
    "SELECT station, date_format(timestamp,'EE') as weekday, hour(timestamp) as hour, COUNT(*) as total_count \
    FROM register \
    WHERE NOT (used_slots = 0 AND free_slots = 0) \
    GROUP BY station, weekday, hour"
).createOrReplaceTempView("total")

In [4]:
threshold = 0.3

final_output_df = ss.sql(
    f"SELECT t.station as station_id, t.weekday, t.hour, critical_count/total_count AS criticality, longitude, latitude \
    FROM critical AS c, total AS t, stations AS s \
    WHERE \
        c.station = t.station AND c.weekday = t.weekday AND c.hour = t.hour \
        AND s.id = t.station \
        AND (critical_count/total_count) > {threshold} \
    ORDER BY \
        criticality DESC, \
        station_id ASC, \
        weekday ASC, \
        hour ASC"
)

In [5]:
final_output_df.show()

+----------+-------+----+-------------------+---------+---------+
|station_id|weekday|hour|        criticality|longitude| latitude|
+----------+-------+----+-------------------+---------+---------+
|         1|    Thu|   0| 0.4581005586592179| 2.180019|41.397978|
|         1|    Thu|   1| 0.4329608938547486| 2.180019|41.397978|
|         1|    Sun|   4|  0.403899721448468| 2.180019|41.397978|
|         1|    Wed|  23|  0.388086642599278| 2.180019|41.397978|
|         1|    Thu|   2|0.38341968911917096| 2.180019|41.397978|
|         1|    Tue|   0| 0.3743016759776536| 2.180019|41.397978|
|         1|    Wed|  22|0.37122557726465366| 2.180019|41.397978|
|         1|    Mon|   1| 0.3659217877094972| 2.180019|41.397978|
|         1|    Wed|   0|0.34108527131782945| 2.180019|41.397978|
|         1|    Mon|   0| 0.3380281690140845| 2.180019|41.397978|
|         1|    Tue|   1| 0.3352272727272727| 2.180019|41.397978|
|         1|    Fri|   0| 0.3307291666666667| 2.180019|41.397978|
|         

In [ ]:
final_output_df.write.csv("output/", header=True)